# Beyond Keywords — Interactive News Classifier

**Demo notebook | Business vs. Science & Technology**

This notebook creates a local Gradio web interface for **new English news headlines**.
It **never trains, fine-tunes, refits, or evaluates** the classifiers.

### How it gets predictions

It does **not** need to open or execute `Beyond_Keywords.ipynb`. Instead, the
completed training notebook saves two reusable model files:

- `outputs/artifacts/tfidf.joblib` — fitted TF-IDF vectorizer **and** Logistic Regression pipeline.
- `outputs/artifacts/sbert_lr.joblib` — fitted Logistic Regression classifier for sentence embeddings.

For the SBERT option, the demo also loads the original **frozen**
`sentence-transformers/all-MiniLM-L6-v2` encoder at the exact pinned revision
used during training. Its weights may need to be downloaded on first use.

**Place this notebook in the same project folder as `Beyond_Keywords.ipynb`**
and keep the `outputs/artifacts/` folder there. That folder is generated by
the original notebook after a successful training run; it is **not included**
in the public GitHub repository by default. If the two `.joblib` files are
missing, ask for the previously saved copies or complete the main training
notebook once before using this demo.

### Requirements

- Use the **same Python environment** as the training notebook (to keep scikit-learn and joblib versions compatible).
- Install Gradio **once** in that environment if needed: `%pip install gradio`.
- Existing packages: `joblib`, `scikit-learn`, `sentence-transformers`, and `torch`.
- Internet is needed **only** for the initial download of pretrained MiniLM weights, if not already cached.

Keep the main notebook's original **Try it** cell unchanged for the course submission.
This is an **optional demonstration interface**, not a replacement for training or evaluation.

## 1. Setup and saved-artifact check

Run this cell first. It only checks local paths; it does not load or change
the models, and does not retrain anything.

In [ ]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().resolve()
ARTIFACTS_DIR = PROJECT_ROOT / "outputs" / "artifacts"

LABELS = ("Business", "Science & Technology")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

MODEL_PATHS = {
    "tfidf": ARTIFACTS_DIR / "tfidf.joblib",
    "sbert": ARTIFACTS_DIR / "sbert_lr.joblib",
}

print("Project directory:", PROJECT_ROOT)
for model_name, file_path in MODEL_PATHS.items():
    print(f"{model_name.upper()} saved classifier:",
          "FOUND" if file_path.is_file() else f"MISSING — {file_path}")

if not all(path.is_file() for path in MODEL_PATHS.values()):
    print("\nThe interface can still open, but predictions require the saved model files.")
    print("Copy the original outputs/artifacts directory beside this notebook.")

## 2. Inference-only Predictor

This class matches the training notebook's saved-model paths, label mapping,
encoder revision, maximum sequence length, and L2 normalization.

**No `.fit()` or training calls appear here.** The Logistic Regression
classifiers are loaded from disk, and the pretrained encoder is loaded
lazily when SBERT is selected for the first time.

In [ ]:
class Predictor:
    def __init__(self):
        self.models = {}
        self.encoder = None

    def _load_encoder(self):
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(
            MODEL_NAME,
            revision=MODEL_REVISION,
            device="cpu",
        )
        model.max_seq_length = 256
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad = False
        return model

    def predict(self, headline: str, method: str = "tfidf") -> str:
        if not isinstance(headline, str) or not headline.strip():
            return "Please enter a non-empty English news headline."

        if method not in MODEL_PATHS:
            return "Please select TF-IDF or Sentence-BERT."

        model_file = MODEL_PATHS[method]
        if not model_file.is_file():
            raise FileNotFoundError(
                f"Missing {model_file.name}. Place the saved training artifacts "
                "in outputs/artifacts beside this notebook."
            )

        if method not in self.models:
            # Only load the previously fitted model; never call fit().
            self.models[method] = joblib.load(model_file)

        features = [headline]
        if method == "sbert":
            if self.encoder is None:
                self.encoder = self._load_encoder()
            features = self.encoder.encode(
                [headline],
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_numpy=True,
            )
            if features.shape != (1, 384):
                raise ValueError("Unexpected sentence embedding size.")

        label_id = int(self.models[method].predict(features)[0])
        if label_id not in (0, 1):
            raise ValueError(f"Unexpected prediction label: {label_id}")
        return LABELS[label_id]


demo_predictor = Predictor()
print("Prediction class loaded. No training was performed.")

## 3. Launch the web interface

Run this cell to display a local website **inside Jupyter**. A localhost
address may also appear; the notebook kernel must remain running for the
interface to work. `share=False` intentionally avoids creating a public link.

The example headlines are **demo inputs**, not additional test data or
reported evaluation scores.

In [ ]:
try:
    import gradio as gr
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Gradio is missing. In this notebook's Python environment, "
        "run `%pip install gradio` once, then rerun this cell."
    ) from exc


def classify_for_demo(headline, method):
    try:
        return demo_predictor.predict(headline, method)
    except FileNotFoundError as exc:
        return f"Model files unavailable: {exc}"
    except Exception as exc:
        return f"Prediction could not be completed: {type(exc).__name__}: {exc}"


with gr.Blocks(
    title="Beyond Keywords | News Topic Classifier",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
) as demo:
    gr.Markdown(
        "# Beyond Keywords\n"
        "### Business vs. Science & Technology\n"
        "Enter a new English news headline and choose the classification method."
    )

    headline_input = gr.Textbox(
        label="News headline",
        placeholder="e.g., A software company reports record quarterly profits",
        lines=2,
    )
    method_input = gr.Radio(
        choices=[
            ("TF-IDF + Logistic Regression", "tfidf"),
            ("Sentence-BERT + Logistic Regression", "sbert"),
        ],
        value="tfidf",
        label="Choose a model",
    )
    classify_button = gr.Button("Classify headline", variant="primary")
    output_label = gr.Textbox(
        label="Predicted category", interactive=False, lines=1
    )

    classify_button.click(
        fn=classify_for_demo,
        inputs=[headline_input, method_input],
        outputs=output_label,
    )
    headline_input.submit(
        fn=classify_for_demo,
        inputs=[headline_input, method_input],
        outputs=output_label,
    )

    gr.Examples(
        examples=[
            ["A software company reports record quarterly profits", "tfidf"],
            ["Scientists discover new evidence of water on Mars", "sbert"],
        ],
        inputs=[headline_input, method_input],
        label="Try an example",
    )

    gr.Markdown(
        "*Research prototype: only Business and Science & Technology "
        "are supported. Out-of-scope headlines will still receive one "
        "of these two labels. Predictions are not evaluation scores.*"
    )

demo.launch(share=False, inline=True, inbrowser=False, show_error=True)

## 4. Test the demo safely

1. Test the TF-IDF option first. It should load `tfidf.joblib` and return one of the two labels.
2. Test the Sentence-BERT option. On its first use, loading pretrained weights may take longer.
3. Clear the headline and click **Classify**. The page should request a non-empty English headline.
4. **Do not** treat these example predictions as test accuracy or Macro F1.

**If no model files are found:** the original notebook must have generated
`outputs/artifacts/tfidf.joblib` and `outputs/artifacts/sbert_lr.joblib`.
They can be **copied from the completed training run** without retraining.
Keep their scikit-learn version compatible with the demo environment.

### Original sources

- [Beyond Keywords training repository](https://github.com/shudiAI/Beyond_Keyword_-NLP_Project)
- [Sentence Transformers model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [Gradio documentation](https://www.gradio.app/guides/quickstart)